# 00_create_sample_dataset_for_embedding

A) Create sample exactly 1000 lines from each month while maintaining the compressed format, you can use the following script.
This code uses np.random.choice to pick random indices and then applies that same mask across all arrays within the .npz file to ensure the data stays aligned (i.e., the ID still matches the summary).

B) Section B we remove core public sector jobs from Onet v30.0 reducing from 894 to 861 (drop 33 core public sector jobs).

In [ ]:
# lets transfer from previous generated and add domain to role_text in order to enrich it;

In [3]:
se for rodar tem rodar tudo de novo e apague todos os npz do stage2!
#!rm -f /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/**/*.npz


In [3]:
import numpy as np
from pathlib import Path

print("start")

src_root = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/job_ads_texts_by_month_npz")
dst_root = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/llama_summary_1000s")
dst_root.mkdir(parents=True, exist_ok=True)

months = [f"adzuna_month{str(i).zfill(2)}" for i in range(1, 15)]

def prepend_domain(text_arr, domain_arr):
    out = []
    for t, d in zip(text_arr, domain_arr):
        t = "" if t is None else str(t).strip()
        d = "" if d is None else str(d).strip()
        out.append(f"[{d}] {t}" if d else t)
    return np.array(out, dtype=object)

for month in months:
    src_file = src_root / f"{month}.npz"
    dst_file = dst_root / f"{month}.npz"

    if not src_file.exists():
        print(f"Skipping {month}: File not found at {src_file}")
        continue

    with np.load(src_file, allow_pickle=True, mmap_mode="r") as data:
        keys = list(data.files)
        n_total = len(data[keys[0]])
        n_sample = min(1000, n_total)

        idx = np.random.choice(n_total, n_sample, replace=False)
        idx.sort()

        sampled = {k: data[k][idx] for k in keys}

        # prepend domain to role_text and taskskill_text
        if "domains" in sampled:
            if "role_text" in sampled:
                sampled["role_text"] = prepend_domain(sampled["role_text"], sampled["domains"])
                print("  domain prepended to role_text")

            if "taskskill_text" in sampled:
                sampled["taskskill_text"] = prepend_domain(sampled["taskskill_text"], sampled["domains"])
                print("  domain prepended to taskskill_text")

        np.savez_compressed(dst_file, **sampled)

        def nonempty_share(arr):
            x = arr[:min(200, len(arr))]
            return sum(bool(str(v).strip()) for v in x) / len(x)

        text_fields = [k for k in ["role_text", "taskskill_text"] if k in sampled]
        shares = {k: round(nonempty_share(sampled[k]), 3) for k in text_fields}

        print(f"Processed {month}: sampled {n_sample}/{n_total} rows -> {dst_file.name}")
        print(f"  keys={keys[:6]}{'...' if len(keys)>6 else ''}")
        if shares:
            print(f"  non-empty shares (first 200 of sample): {shares}")


start
  domain prepended to role_text
  domain prepended to taskskill_text
Processed adzuna_month01: sampled 1000/2657586 rows -> adzuna_month01.npz
  keys=['job_ids', 'role_text', 'taskskill_text', 'domains']
  non-empty shares (first 200 of sample): {'role_text': 1.0, 'taskskill_text': 1.0}
  domain prepended to role_text
  domain prepended to taskskill_text
Processed adzuna_month02: sampled 1000/2229436 rows -> adzuna_month02.npz
  keys=['job_ids', 'role_text', 'taskskill_text', 'domains']
  non-empty shares (first 200 of sample): {'role_text': 1.0, 'taskskill_text': 1.0}
  domain prepended to role_text
  domain prepended to taskskill_text
Processed adzuna_month03: sampled 1000/2303805 rows -> adzuna_month03.npz
  keys=['job_ids', 'role_text', 'taskskill_text', 'domains']
  non-empty shares (first 200 of sample): {'role_text': 1.0, 'taskskill_text': 1.0}
  domain prepended to role_text
  domain prepended to taskskill_text
Processed adzuna_month04: sampled 1000/2331704 rows -> adzuna

In [4]:
import numpy as np
from pathlib import Path
import random

dst_root = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/llama_summary_1000s")

months = [f"adzuna_month{str(i).zfill(2)}" for i in range(1, 15)]

def nonempty_share(arr):
    x = arr
    return sum(bool(str(v).strip()) for v in x) / len(x) if len(x) else 0.0

def is_prefixed(text, domain):
    t = "" if text is None else str(text).strip()
    d = "" if domain is None else str(domain).strip()
    if not d:
        return True  # nothing to check
    return t.startswith(f"[{d}]")

print("Evaluating sampled outputs...\n")

for month in months:
    p = dst_root / f"{month}.npz"
    if not p.exists():
        print(f"{month}: MISSING -> {p}")
        continue

    d = np.load(p, allow_pickle=True, mmap_mode="r")
    keys = list(d.files)

    need = ["job_ids", "role_text", "taskskill_text", "domains"]
    missing = [k for k in need if k not in keys]
    if missing:
        print(f"{month}: missing keys {missing} (has {keys})")
        continue

    n = len(d["job_ids"])
    idxs = random.sample(range(n), min(200, n))

    role_ne = nonempty_share([d["role_text"][i] for i in idxs])
    task_ne = nonempty_share([d["taskskill_text"][i] for i in idxs])
    dom_ne  = nonempty_share([d["domains"][i] for i in idxs])

    role_pref_ok = sum(is_prefixed(d["role_text"][i], d["domains"][i]) for i in idxs) / len(idxs)
    task_pref_ok = sum(is_prefixed(d["taskskill_text"][i], d["domains"][i]) for i in idxs) / len(idxs)

    role_len = [len(str(d["role_text"][i]).strip()) for i in idxs]
    task_len = [len(str(d["taskskill_text"][i]).strip()) for i in idxs]

    print(f"{month}: n={n}")
    print(f"  non-empty share (sample200): role_text={role_ne:.3f} taskskill_text={task_ne:.3f} domains={dom_ne:.3f}")
    print(f"  prefix correctness (sample200): role_text={role_pref_ok:.3f} taskskill_text={task_pref_ok:.3f}")
    print(f"  length stats (sample200): role_text p50={int(np.median(role_len))} p95={int(np.quantile(role_len,0.95))} | taskskill_text p50={int(np.median(task_len))} p95={int(np.quantile(task_len,0.95))}")
    print(f"  example:")
    j = idxs[0]
    print(f"    domain: {d['domains'][j]}")
    print(f"    role_text: {str(d['role_text'][j])[:200]}")
    print(f"    taskskill_text: {str(d['taskskill_text'][j])[:200]}")
    print()


Evaluating sampled outputs...

adzuna_month01: n=1000
  non-empty share (sample200): role_text=1.000 taskskill_text=1.000 domains=1.000
  prefix correctness (sample200): role_text=1.000 taskskill_text=1.000
  length stats (sample200): role_text p50=87 p95=146 | taskskill_text p50=356 p95=497
  example:
    domain: Domestic Services
    role_text: [Domestic Services] Flexible Cleaner Jobs in Gloucester
    taskskill_text: [Domestic Services] Cleaning local homes Providing domestic help Working self-employed Managing flexible hours Transforming customers' houses Self-motivation Time management Communication Physical sta

adzuna_month02: n=1000
  non-empty share (sample200): role_text=1.000 taskskill_text=1.000 domains=1.000
  prefix correctness (sample200): role_text=1.000 taskskill_text=1.000
  length stats (sample200): role_text p50=81 p95=148 | taskskill_text p50=351 p95=489
  example:
    domain: Hospitality & Catering
    role_text: [Hospitality & Catering] Customer Service Assistan

# Out of scope occupations in onet 

For the transformer model in the onet v30.0 positions, I drop 40 non-private, core state roles (e.g. magistrates, police). These aren’t part of the market job space and were adding noise.

Statutory vs. Market Roles

Key Statutory Categories dropped: Judicial, Policing/Security, Emergency Services, Corrections, Postal Monopolies, and Sovereign Infrastructure.

Total exclusions 33 out of 894 onet v30.0 occupations

Final set is 861 occupations.

In [5]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path(
    "/projects/a5u/adu_dev/aisi-economy-index/"
    "aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/embeddings/"
)

RUN_DIR = BASE_DIR / "run_embed_onet_gte"
RUN_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = BASE_DIR / "standard_df_onet_occupations_description_activities_and_tasks.csv"

# Open CSV
df = pd.read_csv(CSV_PATH)

# Quick sanity checks
print(df.shape)
print(df.columns.tolist())
print(df.head(3))


(861, 4)
['O*NET-SOC Code', 'Title', 'Job Role Description', 'Work Activities/Tasks/Skills']
  O*NET-SOC Code                            Title  \
0     11-1011.00                 Chief Executives   
1     11-1011.03    Chief Sustainability Officers   
2     11-1021.00  General and Operations Managers   

                                Job Role Description  \
0  Chief Executives - Determine and formulate pol...   
1  Chief Sustainability Officers - Communicate an...   
2  General and Operations Managers - Plan, direct...   

                        Work Activities/Tasks/Skills  
0  Chief Executives - 'Administer programs for se...  
1  Chief Sustainability Officers - 'Conduct risk ...  
2  General and Operations Managers - 'Develop or ...  


In [8]:
from pathlib import Path
import urllib.request

dst_root = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/embeddings/")
dst = dst_root / "db_30_0_text.zip"

if not dst.exists():
    dst_root.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(
        "https://www.onetcenter.org/dl_files/database/db_30_0_text.zip",
        dst
    )
from zipfile import ZipFile
import pandas as pd

with ZipFile(dst) as z:
    onet_soc_codes = set(
        pd.read_csv(
            z.open("db_30_0_text/Abilities.txt"),
            sep="\t",
            dtype=str
        )["O*NET-SOC Code"]
    )
standard_df_onet_occupations_description_activities_and_tasks = pd.read_csv( dst_root / "standard_df_onet_occupations_description_activities_and_tasks.csv")
check_1= set(standard_df_onet_occupations_description_activities_and_tasks['O*NET-SOC Code'])
print('same onet codes for onet 30.0?',len(onet_soc_codes.intersection(check_1)), ' IF RESPONSE is  894 with core public sector occs or 861 dropped core public sector jobs')

from pathlib import Path
import numpy as np

path = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/embeddings/aspectt_vectors.npz")
data = np.load(path,allow_pickle=True)
print(data.files)
print('same onet titles for onet 30.0?',len(set(data['titles']).intersection(set(standard_df_onet_occupations_description_activities_and_tasks['Title']))), ' IF RESPONSE is 894 with core public sector occs or 861 dropped core public sector jobs')

same onet codes for onet 30.0? 861  IF RESPONSE is  894 with core public sector occs or 861 dropped core public sector jobs
['titles', 'columns', 'levels', 'importance']
same onet titles for onet 30.0? 861  IF RESPONSE is 894 with core public sector occs or 861 dropped core public sector jobs


In [ ]:
# if above is not 861 run the 33 positions drop:

In [9]:
import pandas as pd
list_core_public_state_related_jobs = {
  "definition": {
    "core_public_sector": "Roles with statutory authority, coercive power, public monopoly delivery, or security-vetted state functions. These are typically recruited via government portals/appointments rather than commercial job boards like Adzuna."
  },
  "core_not_expected_in_adzuna_from_your_list": {
    "justice_courts_statutory_legal": [
      "Judges, Magistrate Judges, and Magistrates",
      "Administrative Law Judges, Adjudicators, and Hearing Officers",
      "Judicial Law Clerks",
      "Coroners",
      "Bailiffs",
      "Probation Officers and Correctional Treatment Specialists",
      "Court, Municipal, and License Clerks",
      "Tax Examiners and Collectors, and Revenue Agents"
    ],
    "policing_intelligence_border_security": [
      "First-Line Supervisors of Police and Detectives",
      "Detectives and Criminal Investigators",
      "Police Identification and Records Officers",
      "Intelligence Analysts",
      "Fish and Game Wardens",
      "Parking Enforcement Workers",
      "Police and Sheriff's Patrol Officers",
      "Customs and Border Protection Officers",
      "Transit and Railroad Police",
      "Transportation Security Screeners"
    ],
    "fire_emergency_management_public_safety": [
      "Emergency Management Directors",
      "First-Line Supervisors of Firefighting and Prevention Workers",
      "Firefighters",
      "Fire Inspectors and Investigators",
      "Forest Fire Inspectors and Prevention Specialists",
      "Public Safety Telecommunicators"
    ],
    "corrections_detention": [
      "First-Line Supervisors of Correctional Officers",
      "Correctional Officers and Jailers"
    ],
    "postal_service_monopoly": [
      "Postmasters and Mail Superintendents",
      "Postal Service Clerks",
      "Postal Service Mail Carriers",
      "Postal Service Mail Sorters, Processors, and Processing Machine Operators"
    ],
    "government_programs_admin_gatekeeping": [
      "Eligibility Interviewers, Government Programs"
    ],
    "statutory_public_inspection_enforcement": [
      "Government Property Inspectors and Investigators",
    ],
    "sovereign_infrastructure_control": [
      "Air Traffic Controllers",
    ]
  },
  "notes": {
    "not_included_as_core": [
      "Teachers, nurses, clinicians, and university staff can be public sector but do commonly advertise via job boards depending on country and institution.",
      "Compliance, inspection, regulation, and security roles exist in private sector too. Only the explicitly state-linked statutory variants above were tagged core here.",
      "If you want a stricter version, we can add public-school teaching roles to core. If you want a looser version, we can remove court clerks and keep only judges/ALJs."
    ]
  }
}
import pandas as pd

# 1) core titles only (ignore definition + notes)
core_titles = [
    t.strip()
    for group in list_core_public_state_related_jobs["core_not_expected_in_adzuna_from_your_list"].values()
    for t in group
]

# 2) titles in your npz
series_set = set(pd.Series(data["titles"]).dropna().astype(str).str.strip())

# 3) compare
missing = [t for t in core_titles if t not in series_set]
if missing:
    raise ValueError(f"{len(missing)} missing. Examples: {missing[:20]}")
print(f"OK: {len(core_titles)} core titles present in data['titles'].")


OK: 33 core titles present in data['titles'].


In [10]:
# CSV
old_csv_path = dst_root / "standard_df_onet_occupations_description_activities_and_tasks.csv"
new_csv_path = dst_root / "standard_df_onet_occupations_description_activities_and_tasks_full_onet_30_0.csv"

# NPZ
old_npz_path = dst_root / "aspectt_vectors.npz"
new_npz_path = dst_root / "aspectt_vectors_full_onet_30_0.npz"
old_csv_path.rename(new_csv_path)
old_npz_path.rename(new_npz_path)


PosixPath('/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/embeddings/aspectt_vectors_full_onet_30_0.npz')

In [11]:
from pathlib import Path
import numpy as np
import pandas as pd

# --- paths ---
dst_root = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/embeddings/")
csv_path = dst_root / "standard_df_onet_occupations_description_activities_and_tasks_full_onet_30_0.csv"
npz_path = dst_root / "aspectt_vectors_full_onet_30_0.npz"

csv_out = dst_root / "standard_df_onet_occupations_description_activities_and_tasks.csv"
npz_out = dst_root / "aspectt_vectors.npz"

# --- your 40 titles (already validated present) ---
core_set = set(t.strip() for t in core_titles)

# --- load csv ---
df = pd.read_csv(csv_path, dtype=str)
title_col = "Title"

# mask: keep rows NOT in core list
keep_titles = ~df[title_col].fillna("").astype(str).str.strip().isin(core_set)

df_f = df.loc[keep_titles].copy()
df_f.to_csv(csv_out, index=False)

# --- load npz ---
data = np.load(npz_path, allow_pickle=True)

# build keep mask from npz titles
npz_titles = pd.Series(data["titles"]).astype(str).str.strip()
keep_npz = ~npz_titles.isin(core_set)
keep_npz = keep_npz.to_numpy()

# filter every aligned array (same first dimension as titles)
out = {}
n = len(npz_titles)
for k in data.files:
    arr = data[k]
    if hasattr(arr, "shape") and len(arr.shape) > 0 and arr.shape[0] == n:
        out[k] = arr[keep_npz]
    else:
        out[k] = arr  # metadata etc stays untouched

np.savez_compressed(npz_out, **out)

# --- sanity checks ---
print("Dropped from CSV:", int((~keep_titles).sum()), "kept:", len(df_f))
print("Dropped from NPZ:", int((~keep_npz).sum()), "kept:", int(keep_npz.sum()))
print("CSV/NPZ kept titles match?",
      set(df_f[title_col].astype(str).str.strip()) == set(out["titles"].astype(str)))


Dropped from CSV: 0 kept: 861
Dropped from NPZ: 33 kept: 861
CSV/NPZ kept titles match? True


In [12]:
csv_out = dst_root / "standard_df_onet_occupations_description_activities_and_tasks.csv"
npz_out = dst_root / "aspectt_vectors.npz"

standard_df_onet_occupations_description_activities_and_tasks = pd.read_csv(csv_out)
data = npz_out
path = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/embeddings/aspectt_vectors.npz")
data = np.load(path,allow_pickle=True)
print(data.files)
print('same onet titles for onet 30.0?',len(set(data['titles']).intersection(set(standard_df_onet_occupations_description_activities_and_tasks['Title']))) == 861)
print('occupations set size:',len(set(data['titles'])))

['titles', 'columns', 'levels', 'importance']
same onet titles for onet 30.0? True
occupations set size: 861
